In [ ]:
from tensor_product_space import TensorProductSpace, UnivariateSplineSpace
from hierarchical_mesh import HierarchicalMesh
import numpy as np
import scipy.sparse as sp
from cartesian_mesh import CartesianMesh
from hierarchical_space import HierarchicalSpace

In [ ]:
def refine(knots: np.ndarray, p: int, n_times: int=1)->np.ndarray:
    """Given `knots`, returns its dyadic refinement with multiplicity `p+1`
    at the extremities."""
    knots = np.asarray(knots)
    mult_left = np.searchsorted(knots, knots[0], side='right')
    mult_right = len(knots) - np.searchsorted(knots, knots[-1], side='left')
    pad_left = max(0, p + 1 - mult_left)
    pad_right = max(0, p + 1 - mult_right)
    if pad_left > 0 or pad_right > 0:
        knots = np.concatenate((
            np.full(pad_left, knots[0], dtype=knots.dtype),
            knots,
            np.full(pad_right, knots[-1], dtype=knots.dtype)
        ))
    if n_times == 0:
        return knots
    
    # Find indices where the knot value changes
    jump_idx = np.where(knots[1:] > knots[:-1])[0]
    left_vals = knots[jump_idx]
    right_vals =knots[jump_idx + 1]
    num_new_points = (1<<n_times)-1
    fractions = np.linspace(0.,1.,num_new_points+2)[1:-1]
    new_points = left_vals[:, None] + (right_vals - left_vals)[:, None] * fractions[None, :]
    new_points = new_points.ravel()
    insert_positions = np.repeat(jump_idx + 1, num_new_points)
    
    return np.insert(knots, insert_positions, new_points)
        

In [ ]:
knots1 = np.array([-4, -4, -4, 0, 4, 4, 4])
# knots2 = np.array([-1, -1, -1, -0.5, -0.5, 0, 0, 0.5, 0.5, 1, 1, 1.])
#uss1 = UnivariateSplineSpace(degree=2, knots=knots1)
#uss2 = uss1
#tps = TensorProductSpace(2, [uss1, uss2])

In [ ]:
hs = HierarchicalSpace(knots=[knots1, knots1], degrees=[2])

In [ ]:
hs.refine(marked_cells=[2], level=0)
#hs.refine(marked_cells=[3], level=0)

In [ ]:
hs.mesh.plot_cells()

In [ ]:
np.intersect1d(hs.get_children(0, 12)[0], hs.truly_active[1])

In [ ]:
hs.active_functions[1]

In [ ]:
dofmap = hs.build_global_dof_map()

In [ ]:
dofmap

In [ ]:
hs.Bl_minus[1]

In [ ]:
from scipy.interpolate import NdBSpline, BSpline

In [ ]:
hs.evaluate_thb_spline(0.1)

In [ ]:
my_knots = refine(knots=knots1, p=2, n_times=1)
design_mat = BSpline.design_matrix(0.1, my_knots, 2, extrapolate=False)
nnz1, nnz2 = design_mat.nonzero()
design_mat[nnz1, nnz2]

In [ ]:
hs.mesh.aelem_level[2]

In [ ]:
hs.local_multi_level_extraction_operator(0, 0, 0)

In [ ]:
beziers = hs.level_spaces[0].bezier_operators

In [ ]:
point = [[-0.74, 0.78]]
truc = NdBSpline.design_matrix(point, (knots2, knots2), 2).toarray()
truc2 = NdBSpline.design_matrix(point, (knots1, knots1), 2).toarray()
print(np.allclose(beziers[3]@truc[truc!=0.], truc2[truc2!=0.]))

In [ ]:
from scipy.interpolate import BSpline

f = lambda x: 0.1*x**2-0.7
dof_map = hs.build_global_dof_map()
# print(dof_map)
num_dofs = 0
for i in range(hs.nlevels):
    num_dofs+=np.count_nonzero(~np.isin(hs.active_functions[i], hs.Bl_minus[i], assume_unique=True))
print(num_dofs)
M_global = np.zeros((num_dofs, num_dofs))
F_global = np.zeros(num_dofs)
p = hs.degrees[0]
q_pts, q_wts = np.polynomial.legendre.leggauss(p+1)
for l in range(hs.nlevels):
    for element in hs.mesh.aelem_level[l]:
        # print(f'on to element {element} of level {l}.')
        x_min, x_max = hs.mesh.meshes[l].cells[element]
        J_det = (x_max-x_min)/2.
        M_loc = hs.local_multi_level_extraction_operator(element_idx=element, element_level=l, l=l)
        M_b = np.zeros((p+1, p+1))
        F_b = np.zeros(p+1)
        global_indices = hs.element_global_indices(dof_map=dof_map, element_level=l, element_idx=element)
        # print(f'level {l}, element={element}, global_indices = {global_indices}')
        for q_pt, w in zip(q_pts, q_wts):
            x_q = x_min+(q_pt+1.)/2. *(x_max-x_min)
            active_fcts = np.intersect1d(hs.active_functions[l], hs.level_spaces[l].cell_to_basis([element]))
            b_vals = BSpline.design_matrix(x_q, hs.level_spaces[l].spaces[0].knots, p, extrapolate=False)[:, active_fcts].toarray().ravel()
            #print(b_vals)
            M_b+=w*J_det*np.outer(b_vals, b_vals)
            F_b+=w*J_det*b_vals*f(x_q)
        M_e = M_loc@M_b@M_loc.T
        F_e = M_loc@F_b

        for i, I in enumerate(global_indices):
            F_global[I]+=F_e[i]
            for j, J in enumerate(global_indices):
                M_global[I, J] +=M_e[i,j]

In [ ]:
c = np.linalg.solve(M_global, F_global)